In [0]:
%sql
-- Intenção: Data Quality Report - registros aprovados e reprovados por regra, em cada execução da Silver
WITH expectativas AS (
    SELECT
        origin.update_id AS execucao,
        timestamp,
        explode(
            from_json(
                details:flow_progress.data_quality.expectations,
                'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
            )
        ) AS e
    FROM event_log('2b3e86ca-191c-4a9b-a8fa-5b5fa06ee254')
    WHERE event_type = 'flow_progress'
      AND details:flow_progress.data_quality.expectations IS NOT NULL
)
SELECT
    MIN(timestamp) AS inicio_execucao,
    e.name AS regra,
    SUM(e.passed_records) AS aprovados,
    SUM(e.failed_records) AS reprovados,
    ROUND(100 * SUM(e.failed_records) / (SUM(e.passed_records) + SUM(e.failed_records)), 3) AS pct_reprovacao
FROM expectativas
GROUP BY
    execucao,
    e.name
ORDER BY
    inicio_execucao,
    reprovados DESC;

-- Conclusão esperada: uma linha por regra e execução; na execução incremental, alvara_sem_data_futura com 201 reprovados

In [0]:
%sql
-- Intenção: duração e estado final de todas as execuções dos pipelines do projeto
WITH eventos AS (
    SELECT 'bronze' AS pipeline, * FROM event_log('2b674b56-c3ff-4059-ab29-bec87f868723')
    UNION ALL
    SELECT 'silver', * FROM event_log('2b3e86ca-191c-4a9b-a8fa-5b5fa06ee254')
    UNION ALL
    SELECT 'silver_cdc', * FROM event_log('10c83724-1f67-448c-a8a1-276d22157ecf')
    UNION ALL
    SELECT 'gold', * FROM event_log('1c5a6100-19c2-40a1-bf4d-265580353eaf')
)
SELECT
    pipeline,
    origin.update_id AS execucao,
    MIN(timestamp) AS inicio,
    MAX(timestamp) AS fim,
    timestampdiff(SECOND, MIN(timestamp), MAX(timestamp)) AS duracao_segundos,
    max_by(details:update_progress.state, timestamp) AS estado_final
FROM eventos
WHERE event_type = 'update_progress'
GROUP BY
    pipeline,
    origin.update_id
ORDER BY
    inicio;

-- Conclusão esperada: execuções em ordem cronológica, com COMPLETED (e FAILED nas tentativas com erro)

In [0]:
%sql
-- Intenção: linhas gravadas por tabela em cada execução (evidência da carga incremental)
WITH eventos AS (
    SELECT 'bronze' AS pipeline, * FROM event_log('2b674b56-c3ff-4059-ab29-bec87f868723')
    UNION ALL
    SELECT 'silver', * FROM event_log('2b3e86ca-191c-4a9b-a8fa-5b5fa06ee254')
)
SELECT
    pipeline,
    origin.flow_name AS tabela,
    MIN(timestamp) AS inicio_execucao,
    SUM(CAST(details:flow_progress.metrics.num_output_rows AS BIGINT)) AS linhas_gravadas
FROM eventos
WHERE event_type = 'flow_progress'
  AND details:flow_progress.metrics.num_output_rows IS NOT NULL
GROUP BY
    pipeline,
    origin.flow_name,
    origin.update_id
ORDER BY
    inicio_execucao,
    pipeline,
    tabela;

-- Conclusão esperada: Bronze com 18.385 (carga com defeito), 37.226 (full refresh) e 74.199 (incremental)

In [0]:
%sql
-- Intenção: extrair do event log apenas os campos de configuração relevantes para o relatório
SELECT
    timestamp,
    details:create_update.config.serverless AS serverless,
    details:create_update.config.photon AS photon,
    details:create_update.config.continuous AS modo_continuo,
    details:create_update.config.edition AS edicao,
    details:create_update.config.configuration['pipelines.verticalAutoscaling.pipelineEnabled'] AS autoscale_vertical,
    COALESCE(
        details:create_update.config.libraries[0].file.path,
        details:create_update.config.libraries[0].glob.include
    ) AS codigo_fonte
FROM event_log('2b674b56-c3ff-4059-ab29-bec87f868723')
WHERE event_type = 'create_update'
ORDER BY
    timestamp DESC
LIMIT 3;

-- Conclusão esperada: serverless true, photon true, modo_continuo false, autoscale_vertical true; código-fonte antigo e novo